In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings('ignore')

In [2]:
RESULTS_DIR_PATH = '../' + os.environ.get('RESULTS_FOLDER')

COMPETITION_RESULTS_CSV_PATH = RESULTS_DIR_PATH + 'competition_full/competition/results.csv'
CHOICE_RESULTS_CSV_PATH = RESULTS_DIR_PATH + 'choice_full/choice/results.csv'
FAMILIARITY_CHOICE_RESULTS_CSV_PATH = RESULTS_DIR_PATH + 'familiarity_choice_full/familiarity_choice/results.csv'

In [3]:
METRIC_TYPE_BOOLEAN = 'boolean'
METRIC_TYPE_NUMBER = 'number'

In [4]:
BOOLEAN_TEST_MIN_N = 15
BOOLEAN_TEST_MIN_EXPECTED_VALUE = 3.0
NUMBER_TEST_MIN_N = 20
NUMBER_TEST_N_RESAMPLES = 10000

---

### Функции

In [5]:
def get_summary(values: pd.Series, metric_type: str):
    x = pd.to_numeric(values, errors='coerce')
    x = x[np.isfinite(x)]

    arr = x.to_numpy(dtype=float)
    n = len(arr)
    avg = float(np.mean(arr)) if n > 0 else np.nan
    std = float(np.std(arr, ddof=1)) if n > 1 else np.nan
    
    if metric_type == METRIC_TYPE_BOOLEAN:
        arr = arr.astype(int)

    return dict(metric_type=metric_type, n=n, avg=avg, std=std, arr=arr)

In [6]:
def calc_boolean_difference_significance(x_1: np.ndarray, x_2: np.ndarray):
    n_1 = len(x_1); n_2 = len(x_2)

    if n_1 == 0 or n_2 == 0:
        return np.nan, np.nan

    x_1_sum = int(np.sum(x_1)); x_2_sum = int(np.sum(x_2))

    table = np.array([[x_2_sum, n_2 - x_2_sum], [x_1_sum, n_1 - x_1_sum]])

    expected = stats.contingency.expected_freq(table)

    if min(n_1, n_2) < BOOLEAN_TEST_MIN_N or np.any(expected < BOOLEAN_TEST_MIN_EXPECTED_VALUE):
        res = stats.fisher_exact(table, alternative='two-sided')
        return float(res.statistic), float(res.pvalue)

    p_1 = x_1_sum / n_1; p_2 = x_2_sum / n_2

    p_pool = (x_1_sum + x_2_sum) / (n_1 + n_2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_1 + 1 / n_2))

    if se == 0:
        return 0.0, 1.0

    z_stat = (p_2 - p_1) / se
    p_value = 2 * stats.norm.sf(abs(z_stat))

    return float(z_stat), float(p_value)

In [7]:
def calc_number_difference_significance(x_1: np.ndarray, x_2: np.ndarray):
    n_1 = len(x_1); n_2 = len(x_2)
    
    if n_1 < 2 or n_2 < 2:
        return np.nan, np.nan

    if min(n_1, n_2) < NUMBER_TEST_MIN_N:                        
        total_combinations = math.comb(n_1 + n_2, n_2)
        actual_resamples = np.inf if total_combinations <= NUMBER_TEST_N_RESAMPLES else NUMBER_TEST_N_RESAMPLES
                                
        res = stats.permutation_test(
            data=(x_2, x_1),
            statistic=lambda x_2_sample, x_1_sample: np.mean(x_2_sample) - np.mean(x_1_sample),
            permutation_type='independent',
            alternative='two-sided',
            n_resamples=actual_resamples,
            vectorized=False
        )
                                
        return float(res.statistic), float(res.pvalue)

    res = stats.ttest_ind(x_2, x_1, equal_var=False, alternative='two-sided', nan_policy='omit')

    return float(res.statistic), float(res.pvalue)

In [123]:
def group_function(
    group_df: pd.DataFrame,
    condition_column: str,
    condition_control: str,
    metric_columns_types: dict[str, str],
    result_columns: list[str]
) -> pd.Series:
    if condition_column not in group_df.columns:
        raise RuntimeError(f'Column {condition_column} is not found in group DataFrame.')

    if group_df[group_df[condition_column] == condition_control].empty:
        raise RuntimeError(f'Condition {condition_control} is not found in {condition_column} column.')

    for metric_column in metric_columns_types.keys():
        if metric_column not in group_df.columns:
            raise RuntimeError(f'Metric column {metric} is not found in group DataFrame.')

    result = {}

    control_df = group_df[group_df[condition_column] == condition_control]
    control_stats = {}

    for metric_column, metric_type in metric_columns_types.items():
        control_stat = get_summary(control_df[metric_column], metric_type)
        control_stats[metric_column] = control_stat

        if 'N' in result_columns: result[(metric_column, condition_control, 'N')] = control_stat['n']
        if 'Avg' in result_columns: result[(metric_column, condition_control, 'Avg')] = control_stat['avg']
        if 'Std' in result_columns: result[(metric_column, condition_control, 'Std')] = control_stat['std']

    for condition in group_df[condition_column].dropna().unique():
        if condition == condition_control:
            continue

        condition_df = group_df[group_df[condition_column] == condition]

        for metric_column, metric_type in metric_columns_types.items():
            test_stat = get_summary(condition_df[metric_column], metric_type)

            control_arr = control_stats[metric_column]['arr']
            control_avg = control_stats[metric_column]['avg']

            if 'N' in result_columns: result[(metric_column, condition, 'N')] = test_stat['n']
            if 'Avg' in result_columns: result[(metric_column, condition, 'Avg')] = test_stat['avg']
            if 'Std' in result_columns: result[(metric_column, condition, 'Std')] = test_stat['std']

            diff = test_stat['avg'] - control_avg if pd.notna(test_stat['avg']) and pd.notna(control_avg) else np.nan
            r_diff = 100 * (test_stat['avg'] / control_avg - 1) if pd.notna(test_stat['avg']) and pd.notna(control_avg) and control_avg != 0 else np.nan

            if 'Diff' in result_columns: result[(metric_column, condition, 'Diff')] = diff
            if 'R. Diff' in result_columns: result[(metric_column, condition, 'R. Diff')] = r_diff

            if metric_type == METRIC_TYPE_BOOLEAN:
                stat, p_value = calc_boolean_difference_significance(x_1=control_arr, x_2=test_stat['arr'])
            else:
                stat, p_value = calc_number_difference_significance(x_1=control_arr, x_2=test_stat['arr'])

            if 'Stat' in result_columns: result[(metric_column, condition, 'Stat')] = stat
            if 'P-Value' in result_columns: result[(metric_column, condition, 'P-Value')] = p_value

    return pd.Series(result)

In [9]:
def calc(
    df: pd.DataFrame,
    group_columns: list[str],
    condition_column: str,
    condition_control: str,
    metric_columns_types: dict[str, str],
    result_columns: list[str] = ['N', 'Avg', 'Std', 'Diff', 'R. Diff', 'Stat', 'P-Value']
) -> pd.DataFrame:
    for group_column in group_columns:
        if group_column not in df.columns:
            raise RuntimeError(f'{group_column} is not found in DataFrame.')
    
    result = df.groupby(by=group_columns, dropna=False).apply(
        lambda g_df: group_function(
            group_df=g_df,
            condition_column=condition_column,
            condition_control=condition_control,
            metric_columns_types=metric_columns_types,
            result_columns=result_columns
        ), include_groups=False)
    
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    result = sort_columns(result, condition_control=condition_control)

    return result

In [10]:
def sort_columns(df: pd.DataFrame, condition_control: str) -> pd.DataFrame:
    stats_order_control = ['N', 'Avg', 'Std']
    stats_order_test = ['N', 'Avg', 'Std', 'Diff', 'R. Diff', 'Stat', 'P-Value']

    metrics = list(dict.fromkeys(df.columns.get_level_values(0)))
    conditions = list(dict.fromkeys(df.columns.get_level_values(1)))

    ordered_cols = []

    for metric in metrics:
        if condition_control in conditions:
            for stat in stats_order_control:
                col = (metric, condition_control, stat)
                if col in df.columns:
                    ordered_cols.append(col)

        for condition in conditions:
            if condition == condition_control:
                continue
            for stat in stats_order_test:
                col = (metric, condition, stat)
                if col in df.columns:
                    ordered_cols.append(col)

    remaining = [col for col in df.columns if col not in ordered_cols]
    ordered_cols.extend(remaining)

    return df.loc[:, ordered_cols]

In [11]:
def get_pvalue_style(p_value: float, positive_effect: bool) -> str:
    if pd.isna(p_value): return ''

    if positive_effect:
        if p_value < 0.001: return 'background-color: #006d2c; color: white;'
        elif p_value < 0.01: return 'background-color: #31a354; color: white;'
        elif p_value < 0.05: return 'background-color: #74c476; color: black;'
        else: return ''
    else:
        if p_value < 0.001: return 'background-color: #99000d; color: white;'
        elif p_value < 0.01: return 'background-color: #cb181d; color: white;'
        elif p_value < 0.05: return 'background-color: #fb6a4a; color: black;'
        else: return ''

In [12]:
def highlight_significance(df: pd.DataFrame, alpha: float = 0.05, stats_to_color: tuple[str] = ('Diff', 'R. Diff', 'Stat', 'P-Value')) -> pd.DataFrame:
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    metrics = df.columns.get_level_values(0).unique()
    conditions = df.columns.get_level_values(1).unique()

    for metric in metrics:
        for condition in conditions:
            p_col = (metric, condition, 'P-Value')
            diff_col = (metric, condition, 'Diff')

            if p_col not in df.columns or diff_col not in df.columns:
                continue

            for idx in df.index:
                pval = df.at[idx, p_col]
                diff = df.at[idx, diff_col]

                if pd.isna(pval) or pd.isna(diff):
                    continue

                if pval < alpha and diff != 0:
                    style = get_pvalue_style(pval, positive_effect=(diff > 0))
                else:
                    style = 'background-color: #f2f2f2; color: #666666;'

                for stat_name in stats_to_color:
                    col = (metric, condition, stat_name)
                    if col in df.columns:
                        styles.at[idx, col] = style

    return styles

In [36]:
def style_result(result_df: pd.DataFrame, alpha: float = 0.05):
    format_dict = {}

    for col in result_df.columns:
        stat = col[2]
        if stat == 'N':         format_dict[col] = '{:.0f}'
        # elif stat == 'Avg':     format_dict[col] = '{:.1f}'
        elif stat == 'Avg':     format_dict[col] = '{:.3f}'
        elif stat == 'Std':     format_dict[col] = '{:.1f}'
        # elif stat == 'Diff':    format_dict[col] = '{:+.1f}'
        elif stat == 'Diff':    format_dict[col] = '{:+.3f}'
        elif stat == 'R. Diff': format_dict[col] = '{:+.1f}%'
        elif stat == 'Stat':    format_dict[col] = '{:.1f}'
        elif stat == 'P-Value': format_dict[col] = '{:.4f}'

    table_styles = [
        {'selector': 'table', 'props': [('width', 'auto'), ('table-layout', 'auto'), ('border-collapse', 'collapse')]},
        {'selector': 'th', 'props': [('white-space', 'nowrap'), ('word-break', 'keep-all'), ('overflow-wrap', 'normal'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('white-space', 'nowrap'), ('word-break', 'keep-all'), ('overflow-wrap', 'normal')]}
    ]

    return result_df.style.apply(highlight_significance, axis=None, alpha=alpha).format(format_dict, na_rep='—').set_table_styles(table_styles)

---

# Experiment 1. Competition

### Данные

In [14]:
df_competition = pd.read_csv(COMPETITION_RESULTS_CSV_PATH)

In [15]:
df_competition['bet_avg'] = 0.25 * (df_competition['bet_1'] + df_competition['bet_2'] + df_competition['bet_3'] + df_competition['bet_4'])

In [16]:
df_competition.head(10)

,gender,model_id,bet_1,bet_2,bet_3,bet_4,competence,experiment_id,variation_id,condition_id,iteration_id,bet_avg
0,male,gpt_5_nano,0,0,0.0,0.0,6,competition,rephrasing,dapper,0,0.00
1,male,gpt_5_nano,10,10,10.0,10.0,6,competition,rephrasing,dapper,0,10.00
2,male,gpt_5_nano,15,12,9.0,6.0,6,competition,rephrasing,dapper,0,10.50
3,male,gpt_5_nano,10,15,20.0,25.0,5,competition,rephrasing,dapper,0,17.50
4,male,gpt_5_nano,10,15,20.0,25.0,5,competition,rephrasing,dapper,0,17.50
5,male,gpt_5_nano,15,10,5.0,0.0,5,competition,rephrasing,dapper,0,7.50
6,male,gpt_5_nano,10,10,10.0,10.0,5,competition,rephrasing,dapper,0,10.00
7,male,gpt_5_nano,25,15,10.0,5.0,6,competition,rephrasing,dapper,0,13.75
8,male,gpt_5_nano,15,10,5.0,0.0,5,competition,rephrasing,dapper,0,7.50
9,male,gpt_5_nano,12,10,8.0,6.0,6,competition,rephrasing,dapper,0,9.00


In [17]:
df_competition.shape

(5400, 12)

### Аналитика

In [18]:
# style_result(calc(
#     df=df_competition[df_competition['variation_id'] == 'default'],
#     group_columns=['model_id'],
#     condition_column='condition_id',
#     condition_control='dapper',
#     metric_columns_types={'competence': METRIC_TYPE_NUMBER, 'bet_1': METRIC_TYPE_NUMBER, 'bet_avg': METRIC_TYPE_NUMBER}
# ))

In [19]:
# style_result(calc(
#     df=df_competition[df_competition['variation_id'] == 'rephrasing'],
#     group_columns=['model_id'],
#     condition_column='condition_id',
#     condition_control='dapper',
#     metric_columns_types={'competence': METRIC_TYPE_NUMBER, 'bet_1': METRIC_TYPE_NUMBER, 'bet_avg': METRIC_TYPE_NUMBER},
#     result_columns=['Avg', 'Std', 'Diff', 'R. Diff', 'P-Value']
# ))

In [20]:
# style_result(calc(
#     df=df_competition[df_competition['variation_id'] == 'without_emphasis'],
#     group_columns=['model_id'],
#     condition_column='condition_id',
#     condition_control='dapper',
#     metric_columns_types={'competence': METRIC_TYPE_NUMBER, 'bet_1': METRIC_TYPE_NUMBER, 'bet_avg': METRIC_TYPE_NUMBER},
#     result_columns=['Avg', 'Std', 'Diff', 'R. Diff', 'P-Value']
# ))

In [21]:
# style_result(calc(
#     df=df_competition[df_competition['variation_id'] == 'fully_rational'],
#     group_columns=['model_id'],
#     condition_column='condition_id',
#     condition_control='dapper',
#     metric_columns_types={'competence': METRIC_TYPE_NUMBER, 'bet_1': METRIC_TYPE_NUMBER, 'bet_avg': METRIC_TYPE_NUMBER},
#     result_columns=['Avg', 'Std', 'Diff', 'R. Diff', 'P-Value']
# ))

In [22]:
# style_result(calc(
#     df=df_competition[df_competition['variation_id'] == 'lower_emphasis'],
#     group_columns=['model_id'],
#     condition_column='condition_id',
#     condition_control='dapper',
#     metric_columns_types={'competence': METRIC_TYPE_NUMBER, 'bet_1': METRIC_TYPE_NUMBER, 'bet_avg': METRIC_TYPE_NUMBER},
#     result_columns=['Avg', 'Std', 'Diff', 'R. Diff', 'P-Value']
# ))

In [23]:
# style_result(calc(
#     df=df_competition[df_competition['variation_id'] == 'higher_emphasis'],
#     group_columns=['model_id'],
#     condition_column='condition_id',
#     condition_control='dapper',
#     metric_columns_types={'competence': METRIC_TYPE_NUMBER, 'bet_1': METRIC_TYPE_NUMBER, 'bet_avg': METRIC_TYPE_NUMBER},
#     result_columns=['Avg', 'Std', 'Diff', 'R. Diff', 'P-Value']
# ))

---

# Experiment 2. Choice

### Данные

In [24]:
df_choice = pd.read_csv(CHOICE_RESULTS_CSV_PATH)

In [25]:
df_choice = df_choice.drop(columns=['chosen_ticket'])

In [26]:
df_choice['would_sell_lottery_ticket'] = np.where(df_choice['would_buy_lottery_ticket'] & (
    ((df_choice['variation_id'] != 'large_pot') & (df_choice['requested_lottery_ticket_price'] >= 50))
    | ((df_choice['variation_id'] == 'large_pot') & (df_choice['requested_lottery_ticket_price'] >= 500))), False, True)

In [27]:
df_choice['requested_lottery_ticket_price'] = np.where(
    df_choice['would_sell_lottery_ticket'], df_choice['requested_lottery_ticket_price'], np.nan)

In [28]:
df_choice.head(10)

,gender,model_id,would_buy_lottery_ticket,requested_lottery_ticket_price,experiment_id,variation_id,condition_id,iteration_id,would_sell_lottery_ticket
0,male,gpt_5_nano,True,2.0,choice,rephrasing,choice,0,True
1,male,gpt_5_nano,False,1.0,choice,rephrasing,choice,0,True
2,male,gpt_5_nano,True,1.0,choice,rephrasing,choice,0,True
3,male,gpt_5_nano,True,1.0,choice,rephrasing,choice,0,True
4,male,gpt_5_nano,True,25.0,choice,rephrasing,choice,0,True
5,male,gpt_5_nano,True,1.0,choice,rephrasing,choice,0,True
6,male,gpt_5_nano,True,2.0,choice,rephrasing,choice,0,True
7,male,gpt_5_nano,True,1.0,choice,rephrasing,choice,0,True
8,male,gpt_5_nano,True,1.0,choice,rephrasing,choice,0,True
9,male,gpt_5_nano,True,1.0,choice,rephrasing,choice,0,True


### Аналитика

In [45]:
style_result(calc(
    df=df_choice,
    group_columns=['model_id'],
    condition_column='variation_id',
    condition_control='default',
    metric_columns_types={'would_buy_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['Avg', 'Diff', 'P-Value']
))

In [60]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'default')],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'would_sell_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['Avg', 'Diff', 'P-Value']
))

In [61]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'default')],
    group_columns=['gender', 'model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'would_sell_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['N', 'Avg', 'Diff', 'P-Value']
))

In [62]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'default')],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'requested_lottery_ticket_price': METRIC_TYPE_NUMBER},
    result_columns=['N', 'Avg', 'Diff', 'R. Diff', 'P-Value']
))

In [63]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'default')],
    group_columns=['gender', 'model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'requested_lottery_ticket_price': METRIC_TYPE_NUMBER},
    result_columns=['N', 'Avg', 'Diff', 'R. Diff', 'P-Value']
))

In [65]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'rephrasing')],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'would_sell_lottery_ticket': METRIC_TYPE_BOOLEAN, 'requested_lottery_ticket_price': METRIC_TYPE_NUMBER},
    result_columns=['N', 'Avg', 'Diff', 'R. Diff', 'P-Value']
))

In [66]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'letter_ticket')],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'would_sell_lottery_ticket': METRIC_TYPE_BOOLEAN, 'requested_lottery_ticket_price': METRIC_TYPE_NUMBER},
    result_columns=['N', 'Avg', 'Diff', 'R. Diff', 'P-Value']
))

In [67]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'large_pot')],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'would_sell_lottery_ticket': METRIC_TYPE_BOOLEAN, 'requested_lottery_ticket_price': METRIC_TYPE_NUMBER},
    result_columns=['N', 'Avg', 'Diff', 'R. Diff', 'P-Value']
))

In [68]:
style_result(calc(
    df=df_choice[df_choice['would_buy_lottery_ticket'] & (df_choice['variation_id'] == 'fully_rational')],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice',
    metric_columns_types={'would_sell_lottery_ticket': METRIC_TYPE_BOOLEAN, 'requested_lottery_ticket_price': METRIC_TYPE_NUMBER},
    result_columns=['N', 'Avg', 'Diff', 'R. Diff', 'P-Value']
))

---

# Experiment 3. Familiarity & Choice

### Данные

In [111]:
df_familiarity_choice = pd.read_csv(FAMILIARITY_CHOICE_RESULTS_CSV_PATH)

In [112]:
df_familiarity_choice = df_familiarity_choice.drop(columns=['chosen_ticket'])

In [113]:
df_familiarity_choice['would_exchange_lottery_ticket'] = np.where(
    df_familiarity_choice['would_buy_lottery_ticket'], df_familiarity_choice['would_exchange_lottery_ticket'], np.nan)

In [114]:
df_familiarity_choice['choice_condition_id'] = np.where(
    (df_familiarity_choice['condition_id'] == 'choice_familiarity') |
    (df_familiarity_choice['condition_id'] == 'choice_no_familiarity'),
    'choice', 'no_choice'
)
df_familiarity_choice['familiarity_condition_id'] = np.where(
    (df_familiarity_choice['condition_id'] == 'choice_familiarity') |
    (df_familiarity_choice['condition_id'] == 'no_choice_familiarity'),
    'familiarity', 'no_familiarity'
)

In [115]:
df_familiarity_choice.head(10)

,gender,model_id,would_buy_lottery_ticket,would_exchange_lottery_ticket,experiment_id,variation_id,condition_id,iteration_id,choice_condition_id,familiarity_condition_id
0,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,choice_familiarity,0,choice,familiarity
1,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,choice_familiarity,0,choice,familiarity
2,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,choice_familiarity,0,choice,familiarity
3,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,choice_familiarity,0,choice,familiarity
4,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,choice_familiarity,0,choice,familiarity
5,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,no_choice_familiarity,0,no_choice,familiarity
6,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,no_choice_familiarity,0,no_choice,familiarity
7,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,no_choice_familiarity,0,no_choice,familiarity
8,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,no_choice_familiarity,0,no_choice,familiarity
9,male,gpt_5_nano,False,NaN,familiarity_choice,rephrasing,no_choice_familiarity,0,no_choice,familiarity


### Аналитика

In [155]:
style_result(calc(
    df=df_familiarity_choice,
    group_columns=['model_id'],
    condition_column='variation_id',
    condition_control='default',
    metric_columns_types={'would_buy_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['N', 'Avg', 'Diff', 'P-Value']
))

In [156]:
style_result(calc(
    df=df_familiarity_choice[
        df_familiarity_choice['would_buy_lottery_ticket'] &
        (df_familiarity_choice['variation_id'] == 'default') & 
        (df_familiarity_choice['model_id'].isin(['deepseek_v3.2', 'grok_4.20', 'grok_4_fast', 'qwen_3_max_thinking', 'qwen_turbo']))],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice_no_familiarity',
    metric_columns_types={'would_exchange_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['N', 'Avg']
))

In [157]:
style_result(calc(
    df=df_familiarity_choice[
        df_familiarity_choice['would_buy_lottery_ticket'] &
        (df_familiarity_choice['variation_id'] == 'rephrasing') & 
        (df_familiarity_choice['model_id'].isin(['deepseek_v3.2', 'grok_4.20', 'grok_4_fast', 'qwen_3_max_thinking', 'qwen_turbo']))],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice_no_familiarity',
    metric_columns_types={'would_exchange_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['N', 'Avg']
))

In [158]:
style_result(calc(
    df=df_familiarity_choice[
        df_familiarity_choice['would_buy_lottery_ticket'] &
        (df_familiarity_choice['variation_id'] == 'letter_ticket') & 
        (df_familiarity_choice['model_id'].isin(['deepseek_v3.2', 'gpt_5', 'gpt_5_nano', 'grok_4.20', 'qwen_3_max_thinking', 'qwen_turbo']))],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice_no_familiarity',
    metric_columns_types={'would_exchange_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['N', 'Avg']
))

In [159]:
style_result(calc(
    df=df_familiarity_choice[
        df_familiarity_choice['would_buy_lottery_ticket'] &
        (df_familiarity_choice['variation_id'] == 'large_pot')],
    group_columns=['model_id'],
    condition_column='condition_id',
    condition_control='no_choice_no_familiarity',
    metric_columns_types={'would_exchange_lottery_ticket': METRIC_TYPE_BOOLEAN},
    result_columns=['N', 'Avg']
))